# 16 · Model 1 profiles, full run (all 56,017 users) with Qwen3.5-9B
Owner: Disha. Run by: Ashley, on her RTX 4090 (24 GB). Same model, commit, prompt (v2) and settings as the
approved pilot (`docs/project_notes/model1_pilot_qwen35_20261122/`).

## Before you start (Ashley)
1. **Python with a CUDA build of PyTorch.** Check with `python -c "import torch; print(torch.cuda.is_available())"`:
   it must print `True`. If not: `pip install torch --index-url https://download.pytorch.org/whl/cu128`.
2. **Google Drive for desktop**, with the shared `298A` folder visible on this computer (under *My Drive*, or
   *Shared drives*; if it was shared with you, use *Add shortcut to My Drive* first). The notebook finds it.
3. **Disk space:** about 25 GB free for the model download (Hugging Face cache) plus 2 GB for the run folder.
4. **Power settings:** set the computer to never sleep while plugged in, and close other programs that use the GPU.
5. Optional: a Hugging Face token in the `HF_TOKEN` environment variable (faster downloads).

## How to run
* **Jupyter:** open this notebook, Kernel → Restart, then Run all. If the first cell says it installed libraries,
  restart the kernel and Run all again.
* **Or from a terminal (keeps running even if the browser closes):**
  `jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 16_model1_profiles_full_qwen35.ipynb --output full_run_log.ipynb`
* **If it stops for any reason** (crash, restart, power cut): just run it again. Finished parts are kept and skipped.

## How long
The pilot ran at 2.6 s per profile on an L4 with batches of up to 8. This run packs more short histories into
each batch, so expect roughly **8–20 hours** on the 4090. The real estimate is printed after the first part
(2,000 users) and kept up to date in `progress.json` on the shared Drive, so Disha can follow the run from Drive.

## Where things go
| Where | What |
|---|---|
| Run folder on this computer (`~/298A_full_run/full_v2_qwen35_9b/`) | everything is written here first |
| Shared Drive `298A/model1_profiles/full_v2_qwen35_9b/` | a copy of each finished part, `progress.json`, and at the end all final files |

Final files: `user_profiles.parquet`, `profiles_minilm.npz`, `profiles_e5_large.npz` (agreed profile format),
`profiles_meta.json`, `run_report.json`, `failed_users.json`, `profile_flags.csv`, `checksums.json`, and
`chunks/` (one profile file per 2,000 users, so Ruchika can start before the run ends).
Nothing goes to Git.

## What it reads (frozen data, never changed)
`sequences.parquet` (`user_idx`, `train_items`, `n_train`), `item_text.parquet` (Title, Brand, Category, Price),
`interactions.parquet` (user, item, rating, split). Files are copied to the run folder and must match the frozen
manifest hashes. Before generating, every product in every prompt is checked against `interactions.parquet`:
it must be one of that user's **training** rows, never their validation or test item.

In [1]:
# ============================== SETTINGS (normally nothing to change) ==============================
PROJECT_ROOT = r"G:\.shortcut-targets-by-id\1K2gvVTjU-77GWGMI14ixFt5F57IaijZB\298A"     # the shared '298A' folder on this computer; None = find it automatically
                        # e.g. r'G:\My Drive\298A'  or  '/Users/ashley/Library/CloudStorage/GoogleDrive-.../My Drive/298A'
WORK_DIR = None         # local run folder; None = ~/298A_full_run (on Colab: /content/298A_full_run)

TEACHER_MODEL = 'Qwen/Qwen3.5-9B'
TEACHER_REVISION = 'c202236235762e1c871ad0ccb60c8ee5ba337b9a'   # pinned, same as the pilot
ENABLE_THINKING = False
MAX_NEW_TOKENS = 160
TOKEN_BUDGET = 7000     # prompt + new tokens per batch (padded); lowered automatically after an out-of-memory error
MAX_BATCH = 32
CHUNK_SIZE = 2000       # users per saved part

ENCODERS = ['minilm', 'e5_large']
ENCODER_REVISIONS = {'minilm': '1110a243fdf4706b3f48f1d95db1a4f5529b4d41',
                     'e5_large': '3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3'}
ENCODE_DEVICE = 'cpu'   # encoders run on the CPU after each part, so they never take GPU memory from the teacher

PROMPT_VERSION = 'v2'
EXPECTED_PROMPT_SHA256 = 'a18472f1aefb1a5f3b6397d20d6dc2335a382a6b69c681b908e3e23ba97e9727'   # the approved pilot's
RUN_NAME = 'full_v2_qwen35_9b'
PILOT_RUN = 'pilot_v2_qwen35_9b'   # the approved pilot, used for a quick check that this computer gives the same profiles
MAX_ITEMS = 50
SEED = 42
N_USERS_FULL = 56017
# ====================================================================================================

In [2]:
# Check PyTorch + GPU, then install the pinned library versions (PyTorch itself is never installed or changed here).
import sys, os, subprocess, importlib.metadata as md
def _ver(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None
if _ver('torch') is None:
    raise RuntimeError('PyTorch is not installed. Install the CUDA build first, e.g.\n'
                       '  pip install torch --index-url https://download.pytorch.org/whl/cu128\nthen Run all again.')
PINNED = {'transformers': '5.24.0', 'sentence-transformers': '6.3.0'}
need = [f'{n}=={v}' for n, v in PINNED.items() if _ver(n) != v]
for pkg in ('accelerate', 'pandas', 'pyarrow'):
    if _ver(pkg) is None:
        need.append(pkg)
if need:
    print('Installing', need, '...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *need], check=True)
    if any(m in sys.modules for m in ('transformers', 'sentence_transformers', 'huggingface_hub', 'tokenizers')):
        raise RuntimeError('Installed new library versions. RESTART the kernel now, then Run all again.')
import torch
if not torch.cuda.is_available() and os.environ.get('PROFILES_ALLOW_CPU') != '1':
    raise RuntimeError(f'PyTorch {torch.__version__} cannot see a GPU. Install the CUDA build of PyTorch:\n'
                       '  pip install --force-reinstall torch --index-url https://download.pytorch.org/whl/cu128')
print({n: _ver(n) for n in ('torch', 'transformers', 'sentence-transformers', 'huggingface-hub', 'accelerate', 'pandas')})

Installing ['transformers==5.24.0', 'sentence-transformers==6.3.0', 'accelerate', 'pandas', 'pyarrow'] ...
{'torch': '2.11.0+cu128', 'transformers': '5.24.0', 'sentence-transformers': '6.3.0', 'huggingface-hub': '2.4.0', 'accelerate': '1.16.0', 'pandas': '3.1.1'}


## 1. Locations, disk space and GPU

In [3]:
import json, re, time, hashlib, datetime, random, glob, shutil, gc
from pathlib import Path
import numpy as np
import pandas as pd

for k in ('PROJECT_ROOT', 'WORK_DIR', 'TEACHER_MODEL', 'TEACHER_REVISION', 'RUN_NAME'):   # local testing only
    if os.environ.get(f'PROFILES_{k}'):
        globals()[k] = os.environ[f'PROFILES_{k}']
if os.environ.get('PROFILES_CHUNK_SIZE'):
    CHUNK_SIZE = int(os.environ['PROFILES_CHUNK_SIZE'])

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DATA_REL = Path('04_Data') / 'NewDataRunFiles' / 'processed'
def find_project_root():
    # look for <...>/298A/04_Data/NewDataRunFiles/processed/sequences.parquet in the usual Google Drive locations
    home = Path.home()
    bases = [Path('/content/drive/MyDrive'), *Path('/content/drive/Shareddrives').glob('*')]
    bases += [Path(f'{d}:/') / sub for d in 'DEFGHIJKLMNOPQRSTUVWXYZ' for sub in ('My Drive', 'Shared drives/*')]
    bases += [*home.glob('Library/CloudStorage/GoogleDrive-*/My Drive'), *home.glob('Library/CloudStorage/GoogleDrive-*/Shared drives/*'),
              home / 'Google Drive' / 'My Drive', home / 'GoogleDrive', home / 'gdrive', home / 'Google Drive', home]
    found = []
    for b in bases:
        for base in (glob.glob(str(b)) if '*' in str(b) else [str(b)]):
            for cand in (Path(base) / '298A', *Path(base).glob('*/298A')):
                try:
                    if (cand / DATA_REL / 'sequences.parquet').exists():
                        found.append(cand.resolve())
                except OSError:
                    pass
    return sorted(set(found))

if PROJECT_ROOT is None:
    roots = find_project_root()
    if len(roots) > 1 and len({(r / DATA_REL / 'sequences.parquet').stat().st_size for r in roots}) == 1:
        print('NOTE: the 298A folder is visible in more than one place (e.g. a shortcut); using', roots[0])
        roots = roots[:1]
    if len(roots) != 1:
        raise FileNotFoundError(
            'Could not pick the shared 298A folder automatically. Found: ' + (', '.join(map(str, roots)) or 'none') +
            '.\nSet PROJECT_ROOT in the settings cell to the 298A folder (the one that contains 04_Data and model1_profiles).')
    PROJECT_ROOT = roots[0]
    print('Found the shared 298A folder automatically.')
PROJECT_ROOT = Path(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / DATA_REL
assert (DATA_DIR / 'sequences.parquet').exists(), f'frozen data not found in {DATA_DIR}'
DRIVE_OUT = PROJECT_ROOT / 'model1_profiles' / RUN_NAME
PILOT_DIR = PROJECT_ROOT / 'model1_profiles' / PILOT_RUN
assert RUN_NAME != PILOT_RUN and not RUN_NAME.startswith('pilot'), 'never write into a pilot folder'

WORK_DIR = Path(WORK_DIR) if WORK_DIR else (Path('/content') if IN_COLAB else Path.home()) / '298A_full_run'
RUN_DIR = WORK_DIR / RUN_NAME
CHUNK_DIR, INPUT_DIR = RUN_DIR / 'chunks', WORK_DIR / 'inputs'
for d in (CHUNK_DIR, INPUT_DIR, DRIVE_OUT / 'chunks'):
    d.mkdir(parents=True, exist_ok=True)
for d in (RUN_DIR, DRIVE_OUT):                                         # fail now, not after hours of generation
    (d / '.write_test').write_text('ok'); (d / '.write_test').unlink()

from huggingface_hub import constants as hf_constants, scan_cache_dir
HF_CACHE = Path(hf_constants.HF_HUB_CACHE)
HF_CACHE.mkdir(parents=True, exist_ok=True)
def free_gb(p):
    return shutil.disk_usage(p).free / 1e9
cached = False
try:
    cached = any(r.repo_id == TEACHER_MODEL and any(rev.commit_hash == TEACHER_REVISION for rev in r.revisions)
                 for r in scan_cache_dir(HF_CACHE).repos)
except Exception:
    pass
need_gb = 3 if cached or Path(TEACHER_MODEL).exists() else 25
assert free_gb(HF_CACHE) >= need_gb, (f'only {free_gb(HF_CACHE):.0f} GB free at {HF_CACHE}; need {need_gb} GB for the model. '
                                      'Free some space, or set the HF_HOME environment variable to a bigger drive and restart.')
assert free_gb(RUN_DIR) >= 2, f'only {free_gb(RUN_DIR):.1f} GB free at {RUN_DIR}; need 2 GB'

if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    free_b, total_b = torch.cuda.mem_get_info()
    VRAM_GB, FREE_GB = total_b / 1e9, free_b / 1e9
    assert VRAM_GB >= 20, f'{GPU_NAME} has {VRAM_GB:.0f} GB; this run needs a 24 GB GPU (RTX 4090, L4, A100)'
    if FREE_GB < 21:
        print(f'WARNING: only {FREE_GB:.1f} GB of GPU memory is free. Close other programs that use the GPU '
              '(games, video, browsers with many tabs). The run still works but makes smaller batches.')
else:
    GPU_NAME, VRAM_GB, FREE_GB = 'cpu', 0.0, 0.0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Shared 298A folder :', PROJECT_ROOT)
print('Frozen data        :', DATA_DIR)
print('Run folder (local) :', RUN_DIR)
print('Copies to Drive    :', DRIVE_OUT)
print('Model cache        :', HF_CACHE, f'({free_gb(HF_CACHE):.0f} GB free, model {"already downloaded" if cached else "will be downloaded"})')
print(f'GPU                : {GPU_NAME} ({VRAM_GB:.0f} GB, {FREE_GB:.1f} GB free)')

Shared 298A folder : G:\.shortcut-targets-by-id\1K2gvVTjU-77GWGMI14ixFt5F57IaijZB\298A
Frozen data        : G:\.shortcut-targets-by-id\1K2gvVTjU-77GWGMI14ixFt5F57IaijZB\298A\04_Data\NewDataRunFiles\processed
Run folder (local) : C:\Users\endle\298A_full_run\full_v2_qwen35_9b
Copies to Drive    : G:\.shortcut-targets-by-id\1K2gvVTjU-77GWGMI14ixFt5F57IaijZB\298A\model1_profiles\full_v2_qwen35_9b
Model cache        : C:\Users\endle\.cache\huggingface\hub (940 GB free, model will be downloaded)
GPU                : NVIDIA GeForce RTX 4090 (26 GB, 24.2 GB free)


## 2. Copy the frozen inputs to the run folder and check them against the manifest
Reading from a local copy is faster and safer than reading through the Drive app for many hours.

In [4]:
EXPECTED_SHA256 = {
    'sequences.parquet': '0a55f2c375c06594b322d1e23dcd49d975ac7f8e1f7db12e4cc8bea59128186a',
    'item_text.parquet': '39dbd1ac817df3383def7c677fca830aaf36cfe470e1c0612c543486e6b8909d',
    'interactions.parquet': '298cab581ac90b44504d512b6c706f4616ea93affa8e601be58158a8252c4905',
}
MANIFEST_SHA256 = 'e4b7801c0d9ea27d0b39a6af253ac154d6f4a7181d8a7c1dc31d651a050af0e5'
if os.environ.get('PROFILES_EXPECTED_HASHES_JSON'):                     # local testing with synthetic data only
    EXPECTED_SHA256 = json.loads(Path(os.environ['PROFILES_EXPECTED_HASHES_JSON']).read_text())
    MANIFEST_SHA256 = None

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

for name, expected in EXPECTED_SHA256.items():
    local = INPUT_DIR / name
    if not local.exists() or sha256_file(local) != expected:
        shutil.copyfile(DATA_DIR / name, local.with_suffix('.tmp'))
        os.replace(local.with_suffix('.tmp'), local)
    got = sha256_file(local)
    assert got == expected, f'{name} on Drive is not the frozen version (sha256 {got[:12]}, expected {expected[:12]})'
if MANIFEST_SHA256 and (DATA_DIR / 'manifest.json').exists():
    assert sha256_file(DATA_DIR / 'manifest.json') == MANIFEST_SHA256, 'manifest.json on Drive is not the frozen one'
INPUT_SHA256 = dict(EXPECTED_SHA256)
print('PASS inputs copied to', INPUT_DIR, 'and match the frozen manifest')

PASS inputs copied to C:\Users\endle\298A_full_run\inputs and match the frozen manifest


## 3. Load the data, build every prompt input, and check that only training data is used
Independent of the `train_items` list and of any flag: every (user, product) pair that goes into a prompt is
looked up in `interactions.parquet` and must be a `train` row for that user. Each user's exact input gets a hash
(`input_sha256`) for the later provenance audit.

In [5]:
seq = pd.read_parquet(INPUT_DIR / 'sequences.parquet', columns=['user_idx', 'train_items', 'n_train'])
seq['train_items'] = seq.train_items.apply(lambda a: [int(i) for i in a])
assert seq.user_idx.is_unique and (seq.train_items.str.len() == seq.n_train).all() and (seq.n_train > 0).all()

items = pd.read_parquet(INPUT_DIR / 'item_text.parquet', columns=['item_idx', 'item_text'])
assert items.item_idx.is_unique and items.item_text.str.len().gt(0).all()
FIELDS_KEPT = ('Title', 'Brand', 'Category', 'Price')
_FIELD_SPLIT = re.compile(r' \| (?=(?:Title|Brand|Category|Features|Description|Price|Reviews): )')

def short_item_line(item_text, title_words=30):
    parts = {}
    for part in _FIELD_SPLIT.split(item_text):
        key, _, value = part.partition(': ')
        if key in FIELDS_KEPT and value.strip() and value.strip() != '...':
            value = value.strip()
            if key == 'Title':
                words = value.split()
                value = ' '.join(words[:title_words]) + (' ...' if len(words) > title_words else '')
            parts[key] = value
    return ' | '.join(f'{k}: {parts[k]}' for k in FIELDS_KEPT if k in parts)
ITEM_LINE = dict(zip(items.item_idx, items.item_text.map(short_item_line)))
assert all(ITEM_LINE.values()), 'every item needs at least a title'

inter = pd.read_parquet(INPUT_DIR / 'interactions.parquet', columns=['user_idx', 'item_idx', 'rating', 'split'])
inter = inter.astype({'user_idx': 'int64', 'item_idx': 'int64', 'rating': 'float64'})
assert set(inter.split.unique()) <= {'train', 'val', 'test'}
assert not inter.duplicated(['user_idx', 'item_idx']).any(), 'expected one interaction per user-item pair'

users = seq.sort_values('user_idx').reset_index(drop=True)
users['items_used'] = users.train_items.apply(lambda l: l[-MAX_ITEMS:])
users['n_train_items_used'] = users.items_used.str.len()
used = users[['user_idx', 'items_used']].explode('items_used').rename(columns={'items_used': 'item_idx'})
used['item_idx'] = used.item_idx.astype('int64')
used['pos'] = used.groupby('user_idx').cumcount()
check = used.merge(inter, on=['user_idx', 'item_idx'], how='left', validate='one_to_one')
n_missing, n_not_train = int(check.split.isna().sum()), int((check.split.notna() & (check.split != 'train')).sum())
assert n_missing == 0 and n_not_train == 0, \
    f'provenance check failed: {n_missing} prompt items not in interactions, {n_not_train} are validation/test rows'
check = check.sort_values(['user_idx', 'pos'])
users['ratings_used'] = check.groupby('user_idx').rating.apply(list).reindex(users.user_idx).to_numpy()
assert (users.ratings_used.str.len() == users.n_train_items_used).all()
held_out = inter[inter.split != 'train'].groupby('user_idx').item_idx.apply(set)
leaks = sum(bool(set(l) & held_out.get(u, set())) for u, l in zip(users.user_idx, users.items_used))
assert leaks == 0, f'{leaks} users would see their own validation/test item'
N_TRAIN_ROWS, N_HELDOUT_ROWS = int((inter.split == 'train').sum()), int((inter.split != 'train').sum())
del inter, used, check, held_out; gc.collect()

def input_hash(item_ids, ratings):
    return hashlib.sha256(json.dumps({'items': [int(i) for i in item_ids], 'ratings': [float(r) for r in ratings]},
                                     separators=(',', ':')).encode()).hexdigest()
users['input_sha256'] = [input_hash(i, r) for i, r in zip(users.items_used, users.ratings_used)]
if len(users) != N_USERS_FULL and not os.environ.get('PROFILES_EXPECTED_HASHES_JSON'):
    raise AssertionError(f'expected {N_USERS_FULL} users, found {len(users)}')
print(f'PASS all {int(users.n_train_items_used.sum()):,} prompt items are training rows of their own user '
      f'(training rows {N_TRAIN_ROWS:,}, validation/test rows never used {N_HELDOUT_ROWS:,})')
print(f'{len(users):,} users, {len(items):,} items; items per prompt:', users.n_train_items_used.describe().round(1).to_dict())

PASS all 350,510 prompt items are training rows of their own user (training rows 389,365, validation/test rows never used 112,034)
56,017 users, 25,903 items; items per prompt: {'count': 56017.0, 'mean': 6.3, 'std': 7.0, 'min': 3.0, '25%': 3.0, '50%': 4.0, '75%': 6.0, 'max': 50.0}


## 4. The prompt (v2, identical to the approved pilot)

In [6]:
PROMPT_TEMPLATE = '''You are helping a skin-care shop understand what one of its shoppers likes.

Below are the products this shopper rated, from oldest to newest. Each line is the product's catalog information (with a price band only when the catalog has one), followed by the shopper's own star rating (1 to 5 stars).

{item_lines}

Write a short profile of this shopper's skin-care interests in 2 to 4 sentences (at most 100 words).

How to read the ratings:
- 4 or 5 stars: the shopper liked the product.
- 3 stars: neutral. Do not describe it as liked or disliked.
- 1 or 2 stars: the shopper disliked the product. If there are any, name the disliked product types or brands.

What to describe:
- The product types, categories and brands of the liked products.
- Price level, only from the prices actually shown. Many products have no price, so never say "all" or "always" about prices unless every product shows one. If few prices are shown, leave price out.
- Needs or benefits only when the product titles state them.

Rules:
- Use only what the product list shows. Do not invent products, brands, prices, needs or benefits.
- Never say the shopper is not interested in, avoids or does not care about something that appears in the list.
- Do not use vague praise such as "high-quality" or "effective". Say "natural" or "organic" only if most of the listed titles say so.
- Describe products and interests only. Do not guess or state any health condition, age, gender, ethnicity or other personal identity.
- Refer to the person as "this shopper". Do not use lists, headings or any ids.
- Reply with the profile text only.'''
PROMPT_SHA = hashlib.sha256(PROMPT_TEMPLATE.encode()).hexdigest()
assert PROMPT_SHA == EXPECTED_PROMPT_SHA256, 'the prompt is not the approved v2 prompt'

def build_prompt(item_ids, ratings):
    return PROMPT_TEMPLATE.format(item_lines='\n'.join(
        f'{k}. {ITEM_LINE[i]} | Rating: {r:g}/5' for k, (i, r) in enumerate(zip(item_ids, ratings), 1)))
_ex = build_prompt(users.items_used[0], users.ratings_used[0])
assert not re.search(r'(?:Features|Description|Reviews): ', _ex)
print('PASS prompt is the approved v2 prompt:', PROMPT_SHA[:16])

PASS prompt is the approved v2 prompt: a18472f1aefb1a5f


## 5. Load the teacher (Qwen3.5-9B, text only, pinned commit, bf16, thinking off)
The first time, this downloads about 19 GB.

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM
IS_LOCAL_MODEL = Path(TEACHER_MODEL).exists()
CURRENT_MAIN = None
if not IS_LOCAL_MODEL:
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        assert api.model_info(TEACHER_MODEL, revision=TEACHER_REVISION).sha == TEACHER_REVISION
        CURRENT_MAIN = api.model_info(TEACHER_MODEL).sha
    except AssertionError:
        raise
    except Exception as e:                                               # offline but cached: fine
        print('NOTE: could not reach Hugging Face to confirm the commit (', type(e).__name__, '); using the local cache.')
TEACHER_REV = 'local' if IS_LOCAL_MODEL else TEACHER_REVISION
_rev = None if IS_LOCAL_MODEL else TEACHER_REVISION

tok = AutoTokenizer.from_pretrained(TEACHER_MODEL, revision=_rev)
tok.padding_side = 'left'
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
def chat_text(prompt):
    return tok.apply_chat_template([{'role': 'user', 'content': prompt}], tokenize=False, add_generation_prompt=True,
                                   enable_thinking=ENABLE_THINKING)
GEN_PROMPT_TAIL = chat_text('x')[-40:]
assert ENABLE_THINKING is False and chat_text('x').endswith('<think>\n\n</think>\n\n'), \
    f'thinking is NOT off; generation prompt ends with {GEN_PROMPT_TAIL!r}'

if torch.cuda.is_available():
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model = AutoModelForCausalLM.from_pretrained(TEACHER_MODEL, revision=_rev, dtype=DTYPE, device_map={'': 0})
else:
    DTYPE = torch.float32
    model = AutoModelForCausalLM.from_pretrained(TEACHER_MODEL, revision=_rev, dtype=DTYPE)
model.eval()
TEACHER_CLASS = type(model).__name__
assert TEACHER_CLASS == 'Qwen3_5ForCausalLM', f'unexpected model class {TEACHER_CLASS}'
LOAD_NOTE = str(DTYPE)
EOS_IDS = model.generation_config.eos_token_id
EOS_IDS = set(EOS_IDS if isinstance(EOS_IDS, (list, tuple)) else [EOS_IDS]) | {tok.eos_token_id}
TEACHER_TAG = f'{TEACHER_MODEL if not IS_LOCAL_MODEL else "local:" + Path(TEACHER_MODEL).name}@{TEACHER_REV}'
if torch.cuda.is_available():
    print(f'GPU memory after loading: {torch.cuda.memory_allocated() / 1e9:.1f} GB')
print('PASS thinking off | teacher:', TEACHER_TAG, '|', TEACHER_CLASS, '|', LOAD_NOTE)

THINK_RE = re.compile(r'<think>.*?</think>', re.S)
def clean(raw):
    text = THINK_RE.sub(' ', str(raw)).replace('<think>', ' ').replace('</think>', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return re.sub(r'^(profile|shopper profile)\s*:\s*', '', text, flags=re.I).strip().strip('"').strip()

_TEST_OOM_ABOVE = int(os.environ.get('PROFILES_TEST_OOM_ABOVE', '0'))      # local testing only
def generate_batch(prompts):
    # returns [(clean_text, had_think_tag, hit_max_tokens)] in input order
    if _TEST_OOM_ABOVE and len(prompts) > _TEST_OOM_ABOVE:
        raise torch.cuda.OutOfMemoryError('test hook')
    enc = tok([chat_text(p) for p in prompts], return_tensors='pt', padding=True, add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tok.pad_token_id)
    gen = out[:, enc['input_ids'].shape[1]:]
    texts = tok.batch_decode(gen, skip_special_tokens=True)
    raw = tok.batch_decode(gen, skip_special_tokens=False)
    return [(clean(t), '<think>' in r or '</think>' in r, not any(int(x) in EOS_IDS for x in row))
            for t, r, row in zip(texts, raw, gen)]

C:\Users\endle\Downloads\Multi-Agent Recommendation System Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\endle\Downloads\Multi-Agent Recommendation System Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\endle\.cache\huggingface\hub\models--Qwen--Qwen3.5-9B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer 

GPU memory after loading: 17.9 GB
PASS thinking off | teacher: Qwen/Qwen3.5-9B@c202236235762e1c871ad0ccb60c8ee5ba337b9a | Qwen3_5ForCausalLM | torch.bfloat16


## 6. Quick check: does this computer give the same profiles as the approved pilot?
Eight pilot users are regenerated and compared with their pilot profiles. Small wording differences are normal
(a different GPU rounds slightly differently); garbage or very different text means the setup is broken, and the
run stops here before wasting hours.

In [8]:
from difflib import SequenceMatcher
pilot_file = PILOT_DIR / 'user_profiles.parquet'
by_user = users.set_index('user_idx')
if pilot_file.exists():
    pilot = pd.read_parquet(pilot_file).sample(8, random_state=SEED)
    outs = [generate_batch([build_prompt(by_user.items_used[u], by_user.ratings_used[u])])[0] for u in pilot.user_idx]
    sims = [SequenceMatcher(None, a, o[0]).ratio() for a, o in zip(pilot.profile_text, outs)]
    SMOKE = {'compared_with': PILOT_RUN, 'users': 8, 'identical': int(sum(s == 1.0 for s in sims)),
             'mean_similarity': round(float(np.mean(sims)), 3), 'min_similarity': round(float(np.min(sims)), 3)}
    print(SMOKE)
    print('pilot :', pilot.profile_text.iloc[0][:200], '\nnow   :', outs[0][0][:200])
    assert SMOKE['mean_similarity'] >= 0.5, 'profiles are very different from the approved pilot: stop and tell Disha'
else:
    sample = users.sample(8, random_state=SEED)
    outs = [generate_batch([build_prompt(i, r)])[0] for i, r in zip(sample.items_used, sample.ratings_used)]
    ok = sum(o[0].lower().startswith('this shopper') for o in outs)
    SMOKE = {'compared_with': None, 'users': 8, 'start_with_this_shopper': int(ok)}
    print(f'Pilot profiles not found at {pilot_file}; basic check instead:', SMOKE)
    print('example:', outs[0][0][:200])
    assert ok >= 6, 'profiles do not look right (they should start with "This shopper"): stop and tell Disha'
assert not any(o[1] for o in outs), 'thinking text appeared: stop and tell Disha'
print('PASS setup check')

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{'compared_with': 'pilot_v2_qwen35_9b', 'users': 8, 'identical': 2, 'mean_similarity': 0.785, 'min_similarity': 0.375}
pilot : This shopper favors gentle, soothing cleansers and toners, specifically liking Dove, Aveeno, Pond's, and Dickinson's brands. They show a strong preference for products designed for sensitive skin or t 
now   : This shopper favors gentle, soothing cleansers and toners, specifically liking Dove, Aveeno, Pond's, and Dickinson's brands. They show a strong preference for products designed for sensitive skin or t
PASS setup check


## 7. Generate all profiles, part by part
* Each part (2,000 users) is saved to the run folder as soon as it is done, embedded with both encoders, and
  copied to the shared Drive (`chunks/`), so Ruchika can start before the run ends.
* A rerun skips parts already made with the same users, inputs, prompt, model and settings.
* Batches hold several users of similar length (up to `TOKEN_BUDGET` tokens). After an out-of-memory error the
  batch is halved and the budget lowered for the rest of the run.
* A user whose output is empty is recorded with the reason (retried once at the end), never dropped.
* `progress.json` on the shared Drive shows parts done, users done and the time left.

In [9]:
from sentence_transformers import SentenceTransformer
ENCODER_SPECS = {'minilm': ('sentence-transformers/all-MiniLM-L6-v2', ''),
                 'e5_large': ('intfloat/multilingual-e5-large', 'query: ')}
for k in ENCODER_SPECS:                                                  # local testing only
    if os.environ.get(f'PROFILES_ENCODER_{k.upper()}'):
        ENCODER_SPECS[k] = (os.environ[f'PROFILES_ENCODER_{k.upper()}'], ENCODER_SPECS[k][1])
ENC_REV = {n: 'local' if Path(ENCODER_SPECS[n][0]).exists() else ENCODER_REVISIONS[n] for n in ENCODERS}
ENCODER_MODELS = {n: SentenceTransformer(ENCODER_SPECS[n][0], revision=None if ENC_REV[n] == 'local' else ENC_REV[n],
                                         device=ENCODE_DEVICE) for n in ENCODERS}
SETUP = {'prompt_sha': PROMPT_SHA, 'teacher': TEACHER_TAG, 'enable_thinking': ENABLE_THINKING,
         'max_new_tokens': MAX_NEW_TOKENS, 'encoders': {n: f'{ENCODER_SPECS[n][0]}@{ENC_REV[n]}' for n in ENCODERS}}
SETUP_SHA = hashlib.sha256(json.dumps(SETUP, sort_keys=True).encode()).hexdigest()

def embed(texts):
    return {n: ENCODER_MODELS[n].encode([ENCODER_SPECS[n][1] + t for t in texts], batch_size=64, normalize_embeddings=True,
                                        show_progress_bar=False, convert_to_numpy=True).astype(np.float32) for n in ENCODERS}

def write_npz(path, user_idx, emb):
    tmp = path.with_name(path.name + '.tmp.npz')
    np.savez(tmp, user_idx=np.asarray(user_idx, dtype=np.int64), embeddings=emb, source=np.array('llm_profile'),
             train_items_only=np.array(True))
    os.replace(tmp, path)

def copy_to_drive(src, rel):
    # copy one finished file to the shared Drive; a Drive hiccup never stops the run (the final step copies again)
    dst = DRIVE_OUT / rel
    for attempt in range(3):
        try:
            dst.parent.mkdir(parents=True, exist_ok=True)
            tmp = dst.with_name(dst.name + '.partial')
            shutil.copyfile(src, tmp)
            os.replace(tmp, dst)
            return True
        except OSError as e:
            time.sleep(5 * (attempt + 1)); last = e
    print(f'  WARNING: could not copy {rel} to Drive ({last}); it will be retried at the end')
    return False

TOKEN_BUDGET_NOW = TOKEN_BUDGET
_TEST_FAIL_ONCE = {int(x) for x in os.environ.get('PROFILES_TEST_FAIL_USERS', '').split(',') if x}   # local testing only
def make_batches(part):
    lens = part.n_tokens.to_numpy()
    order = np.argsort(-lens, kind='stable')
    batches, cur, cur_max = [], [], 0
    for i in order:
        m = max(cur_max, lens[i])
        if cur and ((len(cur) + 1) * (m + MAX_NEW_TOKENS) > TOKEN_BUDGET_NOW or len(cur) >= MAX_BATCH):
            batches.append(cur); cur, m = [], lens[i]
        cur.append(i); cur_max = m
    if cur:
        batches.append(cur)
    return batches

def run_users(part):
    global TOKEN_BUDGET_NOW
    part = part.reset_index(drop=True)
    prompts = [build_prompt(i, r) for i, r in zip(part.items_used, part.ratings_used)]
    part['n_tokens'] = [len(x) for x in tok([chat_text(p) for p in prompts], add_special_tokens=False)['input_ids']]
    rows, queue = [], make_batches(part)
    while queue:
        idx = queue.pop(0)
        t0, err = time.time(), ''
        try:
            outs = generate_batch([prompts[i] for i in idx])
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if len(idx) > 1:
                TOKEN_BUDGET_NOW = max(1000, int(TOKEN_BUDGET_NOW * 0.75))
                half = len(idx) // 2
                queue[:0] = [idx[:half], idx[half:]]
                print(f'  out of GPU memory with {len(idx)} users -> split; token budget now {TOKEN_BUDGET_NOW}')
                continue
            outs, err = [('', False, False)], 'out of GPU memory'
        except Exception as e:
            outs, err = [('', False, False)] * len(idx), f'{type(e).__name__}: {e}'[:300]
        secs = (time.time() - t0) / len(idx)
        for i, (text, had_think, hit_max) in zip(idx, outs):
            u = part.iloc[i]
            if int(u.user_idx) in _TEST_FAIL_ONCE:
                _TEST_FAIL_ONCE.discard(int(u.user_idx)); text, err = '', 'test hook'
            rows.append(dict(user_idx=int(u.user_idx), profile_text=text, status='ok' if text else 'failed',
                             reason='' if text else (err or 'empty output'), raw_had_think_tag=bool(had_think),
                             hit_max_tokens=bool(hit_max), n_train_items_used=int(u.n_train_items_used),
                             input_sha256=u.input_sha256, seconds=secs, batch_size=len(idx), setup_sha=SETUP_SHA))
    return pd.DataFrame(rows).sort_values('user_idx').reset_index(drop=True)

def chunk_ok(c, part):
    p = CHUNK_DIR / f'chunk_{c:05d}.parquet'
    if not p.exists() or not all((CHUNK_DIR / f'chunk_{c:05d}_{n}.npz').exists() for n in ENCODERS):
        return False
    prev = pd.read_parquet(p)
    return (len(prev) == len(part) and prev.setup_sha.eq(SETUP_SHA).all()
            and set(zip(prev.user_idx, prev.input_sha256)) == set(zip(part.user_idx, part.input_sha256)))

chunks = [users.iloc[s:s + CHUNK_SIZE] for s in range(0, len(users), CHUNK_SIZE)]
STOP_AFTER = int(os.environ['PROFILES_STOP_AFTER_CHUNKS']) if os.environ.get('PROFILES_STOP_AFTER_CHUNKS') else None
RUN_START = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
t_start, done_now, users_now = time.time(), 0, 0

def write_progress(state):
    done = [c for c in range(len(chunks)) if (CHUNK_DIR / f'chunk_{c:05d}.parquet').exists()]
    users_done = sum(len(chunks[c]) for c in done)
    rate = (time.time() - t_start) / users_now if users_now else None
    left_h = round(rate * (len(users) - users_done) / 3600, 1) if rate else None
    prog = {'state': state, 'parts_done': len(done), 'parts_total': len(chunks), 'users_done': users_done,
            'users_total': len(users), 'hours_left_estimate': left_h,
            'seconds_per_profile_this_session': round(rate, 3) if rate else None, 'gpu': GPU_NAME,
            'session_started_utc': RUN_START,
            'updated_utc': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')}
    (RUN_DIR / 'progress.json').write_text(json.dumps(prog, indent=1))
    copy_to_drive(RUN_DIR / 'progress.json', 'progress.json')
    return prog

for c, part in enumerate(chunks):
    if chunk_ok(c, part):
        print(f'part {c + 1}/{len(chunks)}: already done, skipped')
        continue
    if STOP_AFTER is not None and done_now >= STOP_AFTER:
        raise SystemExit(f'test hook: simulated stop after {done_now} part(s)')
    res = run_users(part)
    ok = res.status == 'ok'
    emb = embed(res.profile_text[ok].tolist())
    for n in ENCODERS:
        write_npz(CHUNK_DIR / f'chunk_{c:05d}_{n}.npz', res.user_idx[ok], emb[n])
    tmp = CHUNK_DIR / f'chunk_{c:05d}.parquet.tmp'
    res.to_parquet(tmp, index=False)
    os.replace(tmp, CHUNK_DIR / f'chunk_{c:05d}.parquet')               # the part counts as done only now
    for name in [f'chunk_{c:05d}.parquet'] + [f'chunk_{c:05d}_{n}.npz' for n in ENCODERS]:
        copy_to_drive(CHUNK_DIR / name, f'chunks/{name}')
    done_now += 1; users_now += len(part)
    prog = write_progress('running')
    print(f'part {c + 1}/{len(chunks)}: {len(res)} users, {int((~ok).sum())} failed, '
          f'{(time.time() - t_start) / 3600:.2f} h this session, about {prog["hours_left_estimate"]} h left')
write_progress('generation finished')
print('All parts done.')

C:\Users\endle\Downloads\Multi-Agent Recommendation System Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\endle\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2

part 1/29: 2000 users, 0 failed, 0.73 h this session, about 19.8 h left
part 2/29: 2000 users, 0 failed, 1.49 h this session, about 19.4 h left
part 3/29: 2000 users, 0 failed, 2.69 h this session, about 22.5 h left
part 4/29: 2000 users, 0 failed, 3.83 h this session, about 23.0 h left
part 5/29: 2000 users, 0 failed, 4.71 h this session, about 21.7 h left
part 6/29: 2000 users, 0 failed, 5.56 h this session, about 20.4 h left
part 7/29: 2000 users, 0 failed, 6.40 h this session, about 19.2 h left
part 8/29: 2000 users, 0 failed, 7.23 h this session, about 18.1 h left
part 9/29: 2000 users, 0 failed, 8.05 h this session, about 17.0 h left
part 10/29: 2000 users, 0 failed, 8.88 h this session, about 16.0 h left
part 11/29: 2000 users, 0 failed, 9.72 h this session, about 15.0 h left
part 12/29: 2000 users, 0 failed, 10.56 h this session, about 14.1 h left
part 13/29: 2000 users, 0 failed, 11.40 h this session, about 13.2 h left
part 14/29: 2000 users, 0 failed, 12.24 h this session, ab

## 8. Retry failed users once, then put all parts together

In [10]:
res = pd.concat([pd.read_parquet(CHUNK_DIR / f'chunk_{c:05d}.parquet') for c in range(len(chunks))], ignore_index=True)
assert res.user_idx.is_unique and len(res) == len(users), 'every user must have exactly one result'
assert res.setup_sha.eq(SETUP_SHA).all(), 'all parts must come from the same setup'

retry_path = CHUNK_DIR / 'retry.parquet'
failed_ids = set(res.user_idx[res.status == 'failed'])
if failed_ids:
    if retry_path.exists() and set(pd.read_parquet(retry_path).user_idx) == failed_ids:
        retry = pd.read_parquet(retry_path)
    else:
        print(f'Retrying {len(failed_ids)} failed users one at a time ...')
        TOKEN_BUDGET_NOW = 1                                             # one user per batch
        retry = run_users(users[users.user_idx.isin(failed_ids)])
        retry.to_parquet(retry_path, index=False)
    res = pd.concat([res[~res.user_idx.isin(retry.user_idx[retry.status == 'ok'])], retry[retry.status == 'ok']],
                    ignore_index=True).sort_values('user_idx').reset_index(drop=True)
    retry_ok = retry[retry.status == 'ok']
    if len(retry_ok):
        retry_emb = embed(retry_ok.profile_text.tolist())
        RETRY_VECS = {n: dict(zip(retry_ok.user_idx, retry_emb[n])) for n in ENCODERS}
    else:
        RETRY_VECS = {n: {} for n in ENCODERS}
else:
    RETRY_VECS = {n: {} for n in ENCODERS}

ok = res[res.status == 'ok'].sort_values('user_idx').reset_index(drop=True)
profiles = pd.DataFrame({'user_idx': ok.user_idx.astype('int64'), 'profile_text': ok.profile_text.astype(str),
                         'teacher_model': TEACHER_TAG, 'prompt_version': PROMPT_VERSION,
                         'n_train_items_used': ok.n_train_items_used.astype('int64'),
                         'input_sha256': ok.input_sha256.astype(str)})
failed = [dict(user_idx=int(r.user_idx), reason=r.reason) for r in res[res.status == 'failed'].itertuples()]
assert len(profiles) + len(failed) == len(users)

vecs = {}
for n in ENCODERS:
    parts_u, parts_e = [], []
    for c in range(len(chunks)):
        with np.load(CHUNK_DIR / f'chunk_{c:05d}_{n}.npz', allow_pickle=False) as z:
            parts_u.append(z['user_idx']); parts_e.append(z['embeddings'])
    lookup = dict(zip(np.concatenate(parts_u).tolist(), np.concatenate(parts_e)))
    lookup.update(RETRY_VECS[n])
    vecs[n] = np.stack([lookup[u] for u in profiles.user_idx]).astype(np.float32)
    assert vecs[n].shape[0] == len(profiles)

profiles.to_parquet(RUN_DIR / 'user_profiles.parquet', index=False)
(RUN_DIR / 'failed_users.json').write_text(json.dumps(failed, indent=1))
for n in ENCODERS:
    write_npz(RUN_DIR / f'profiles_{n}.npz', profiles.user_idx.to_numpy(), vecs[n])
print(f'{len(profiles):,} profiles, {len(failed)} failed users')

56,017 profiles, 0 failed users


## 9. Checks on the whole run
Coverage, empty and duplicate profiles, thinking text, cut-off profiles, and the same flags as the pilot.
These are counts for the report and for the separate profile audit; they reject nothing.

In [11]:
words = profiles.profile_text.str.split().str.len()
dup_text = profiles.profile_text.duplicated(keep=False)
SENSITIVE = re.compile(r"\b(she|he|her|his|woman|women|man|men|female|male|girl|boy|teen\w*|age[ds]?|old|young|"
                       r"years|mother|mom|pregnan\w*|baby|wife|husband|asian|black|white|latin\w*|african|"
                       r"diabet\w*|eczema|psoriasis|rosacea|cancer|disease|condition|allerg\w*|acne)\b", re.I)
LOW_MENTION = re.compile(r"dislik|low(er)?[- ]rat|poor|[12][- ]star|[12]/5|did not like|didn't like|less favou?r|negative|"
                         r"less satisf|did not enjoy|didn't enjoy", re.I)
by_user = users.set_index('user_idx')
def sensitive_terms(uid, text):
    blob = ' '.join(ITEM_LINE[i] for i in by_user.items_used[uid]).lower()
    found = {m.lower() for m in SENSITIVE.findall(text)}
    return ', '.join(sorted(t for t in found if not re.search(rf'\b{re.escape(t)}\b', blob)))
res_by = res.set_index('user_idx')
flags = pd.DataFrame({
    'user_idx': profiles.user_idx,
    'words': words,
    'duplicate_text': dup_text,
    'raw_had_think_tag': profiles.user_idx.map(res_by.raw_had_think_tag).astype(bool),
    'hit_max_tokens': profiles.user_idx.map(res_by.hit_max_tokens).astype(bool),
    'over_120_words': words > 120,
    'possible_sensitive_terms': [sensitive_terms(u, t) for u, t in zip(profiles.user_idx, profiles.profile_text)],
    'says_all_about_price': profiles.profile_text.str.contains(r"\b(?:all|every|always)\b[^.]{0,90}(?:\$|price)", case=False),
    'claims_absence': profiles.profile_text.str.contains(r"not (?:a )?(?:major )?concern|no (?:apparent )?interest|not interested|avoids?\b|does not care", case=False),
    'vague_praise': profiles.profile_text.str.contains(r"high[- ]quality|\beffective\b", case=False),
    'dislikes_not_mentioned': [any(r <= 2 for r in by_user.ratings_used[u]) and not LOW_MENTION.search(t)
                               for u, t in zip(profiles.user_idx, profiles.profile_text)],
    'meta_or_reasoning_text': profiles.profile_text.str.contains(r"\blet me\b|\bokay,|\bfirst,? i\b|\bthe user\b|\bi (?:need|will|should)\b", case=False),
})
flags.to_csv(RUN_DIR / 'profile_flags.csv', index=False)
FLAG_COUNTS = {c: int(flags[c].sum()) for c in flags.columns if flags[c].dtype == bool}
FLAG_COUNTS['possible_sensitive_terms'] = int((flags.possible_sensitive_terms != '').sum())
emb_dups = {n: int(len(vecs[n]) - len(np.unique(np.round(vecs[n], 5), axis=0))) for n in ENCODERS}
COVERAGE = {'users': int(len(users)), 'profiles': int(len(profiles)), 'failed': int(len(failed)),
            'empty_profiles': int((profiles.profile_text.str.strip() == '').sum()),
            'duplicate_profile_texts': int(dup_text.sum()), 'duplicate_embeddings': emb_dups}
print('coverage:', COVERAGE)
print(f'words per profile: mean {words.mean():.1f}, median {words.median():.0f}, max {words.max()}')
print('flags:', FLAG_COUNTS)

coverage: {'users': 56017, 'profiles': 56017, 'failed': 0, 'empty_profiles': 0, 'duplicate_profile_texts': 21, 'duplicate_embeddings': {'minilm': 12, 'e5_large': 12}}
words per profile: mean 62.3, median 62, max 104
flags: {'duplicate_text': 21, 'raw_had_think_tag': 0, 'hit_max_tokens': 7, 'over_120_words': 0, 'says_all_about_price': 3164, 'claims_absence': 139, 'vague_praise': 389, 'dislikes_not_mentioned': 1428, 'meta_or_reasoning_text': 33, 'possible_sensitive_terms': 162}


## 10. Check the profile files and write the report and metadata
The `.npz` checks are the same rules as `read_profile_file` in `src/data/profile_file.py`.

In [12]:
for n in ENCODERS:
    with np.load(RUN_DIR / f'profiles_{n}.npz', allow_pickle=False) as z:
        assert set(z.files) == {'user_idx', 'embeddings', 'source', 'train_items_only'}
        uid, emb, src, flag = z['user_idx'], z['embeddings'], z['source'], z['train_items_only']
        assert flag.ndim == 0 and flag.dtype == np.bool_ and bool(flag)
        assert src.ndim == 0 and src.dtype.kind == 'U' and str(src) == 'llm_profile'
        assert uid.ndim == 1 and uid.dtype == np.int64 and len(np.unique(uid)) == len(uid) and uid.min() >= 0
        assert uid.max() < N_USERS_FULL
        assert emb.ndim == 2 and emb.shape[0] == len(uid) and emb.dtype == np.float32 and np.isfinite(emb).all()
        assert np.allclose(np.linalg.norm(emb, axis=1), 1.0, atol=1e-4)
        assert (uid == profiles.user_idx.to_numpy()).all()
    print('OK', n, emb.shape)

timing = res.groupby(pd.cut(res.n_train_items_used, [0, 4, 9, 1000], labels=['3-4', '5-9', '10+'])).seconds.mean()
REPORT = {'coverage': COVERAGE, 'flags': FLAG_COUNTS, 'setup_check': SMOKE,
          'mean_words': round(float(words.mean()), 1), 'median_words': float(words.median()),
          'seconds_per_profile_mean': round(float(res.seconds.mean()), 3),
          'seconds_per_profile_by_items': {k: round(float(v), 3) for k, v in timing.items()},
          'gpu': GPU_NAME, 'teacher': TEACHER_TAG, 'loaded_as': LOAD_NOTE, 'enable_thinking': ENABLE_THINKING,
          'token_budget_start': TOKEN_BUDGET, 'max_batch': MAX_BATCH}
(RUN_DIR / 'run_report.json').write_text(json.dumps(REPORT, indent=2))
meta = {
    'mode': 'full', 'run_name': RUN_NAME, 'finished_utc': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'teacher_model': TEACHER_MODEL if not IS_LOCAL_MODEL else 'local test model', 'teacher_revision': TEACHER_REV,
    'teacher_main_at_run_time': CURRENT_MAIN, 'teacher_class': TEACHER_CLASS, 'teacher_loaded_as': LOAD_NOTE,
    'enable_thinking': ENABLE_THINKING, 'generation_prompt_tail': GEN_PROMPT_TAIL, 'gpu': GPU_NAME,
    'generation': {'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS, 'seed': SEED, 'token_budget': TOKEN_BUDGET,
                   'max_batch': MAX_BATCH},
    'prompt_version': PROMPT_VERSION, 'prompt_template': PROMPT_TEMPLATE, 'prompt_sha256': PROMPT_SHA,
    'item_text_fields_kept': list(FIELDS_KEPT), 'max_items_per_user': MAX_ITEMS, 'items_order': 'oldest to newest',
    'inputs': {'sha256': INPUT_SHA256, 'manifest_sha256': MANIFEST_SHA256,
               'columns_read': {'sequences.parquet': ['user_idx', 'train_items', 'n_train'],
                                'item_text.parquet': ['item_idx', 'item_text'],
                                'interactions.parquet': ['user_idx', 'item_idx', 'rating', 'split']},
               'provenance_check': 'every prompt item is a train row of its own user in interactions.parquet; '
                                   'no prompt contains the user\'s validation or test item',
               'per_user_input_hash': 'sha256 of {"items": items_used, "ratings": ratings_used}, column input_sha256'},
    'selection_rule': 'all users in sequences.parquet', 'n_users': int(len(users)), 'n_profiles': int(len(profiles)),
    'n_failed': int(len(failed)),
    'encoders': {n: dict(model=ENCODER_SPECS[n][0], revision=ENC_REV[n], prefix=ENCODER_SPECS[n][1],
                         dim=int(vecs[n].shape[1]), l2_normalized=True, device=ENCODE_DEVICE,
                         file=f'profiles_{n}.npz', sha256=sha256_file(RUN_DIR / f'profiles_{n}.npz')) for n in ENCODERS},
    'report': REPORT,
    'versions': {n: _ver(n) for n in ('torch', 'transformers', 'sentence-transformers', 'huggingface-hub', 'accelerate',
                                      'numpy', 'pandas', 'pyarrow')},
    'python': sys.version.split()[0],
}
(RUN_DIR / 'profiles_meta.json').write_text(json.dumps(meta, indent=2))
files = sorted(p for p in RUN_DIR.rglob('*') if p.is_file() and p.name not in ('checksums.json', 'progress.json'))
CHECKSUMS = {str(p.relative_to(RUN_DIR)).replace(os.sep, '/'): sha256_file(p) for p in files}
(RUN_DIR / 'checksums.json').write_text(json.dumps(CHECKSUMS, indent=1))
print(json.dumps({k: REPORT[k] for k in ('coverage', 'mean_words', 'seconds_per_profile_mean')}, indent=1))

OK minilm (56017, 384)
OK e5_large (56017, 1024)
{
 "coverage": {
  "users": 56017,
  "profiles": 56017,
  "failed": 0,
  "empty_profiles": 0,
  "duplicate_profile_texts": 21,
  "duplicate_embeddings": {
   "minilm": 12,
   "e5_large": 12
  }
 },
 "mean_words": 62.3,
 "seconds_per_profile_mean": 1.549
}


## 11. Copy everything to the shared Drive and check the copies

In [13]:
bad = []
for rel, h in {**CHECKSUMS, 'checksums.json': sha256_file(RUN_DIR / 'checksums.json')}.items():
    dst = DRIVE_OUT / rel
    if not (dst.exists() and sha256_file(dst) == h):
        copy_to_drive(RUN_DIR / rel, rel)
    if not (dst.exists() and sha256_file(dst) == h):
        bad.append(rel)
write_progress('finished' if not bad else 'finished, Drive copy incomplete')
if bad:
    print('WARNING: these files did not reach Drive:', bad, '\nRun this cell again, or upload the folder', RUN_DIR, 'by hand.')
else:
    print('PASS all', len(CHECKSUMS) + 1, 'files are on the shared Drive and match their checksums:', DRIVE_OUT)
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()
print('\nDone. Tell Disha: profiles', len(profiles), '| failed', len(failed), '| folder', DRIVE_OUT)

PASS all 95 files are on the shared Drive and match their checksums: G:\.shortcut-targets-by-id\1K2gvVTjU-77GWGMI14ixFt5F57IaijZB\298A\model1_profiles\full_v2_qwen35_9b

Done. Tell Disha: profiles 56017 | failed 0 | folder G:\.shortcut-targets-by-id\1K2gvVTjU-77GWGMI14ixFt5F57IaijZB\298A\model1_profiles\full_v2_qwen35_9b
